In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [26]:

class H2GCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes, num_layers):
        super().__init__()

        self.K = num_layers
        # self.A = A  # adjacency matrix (no self loops)
        # self.n = A.shape[0]

        # # Precompute adjacency matrices
        # self.A1 = self.normalize(A)  # 1-hop
        # self.A2 = self.compute_A2(A)  # 2-hop (cleaned)

        # Feature embedding (S1)
        self.embed = nn.Linear(in_dim, hidden_dim)

        # Final classifier
        final_dim = (2 ** (num_layers + 1) - 1) * hidden_dim
        self.classifier = nn.Linear(final_dim, num_classes)

    def normalize(self, A):
        """Symmetric normalization"""
        deg = A.sum(dim=1)
        deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
        D_inv_sqrt = torch.diag(deg_inv_sqrt)
        return D_inv_sqrt @ A @ D_inv_sqrt

    def compute_A2(self, A):
        """Compute clean 2-hop adjacency"""
        A2 = A @ A

        # Remove self and 1-hop
        I = torch.eye(A.shape[0], device=A.device)
        A2 = A2 - A - I

        # Binarize
        A2 = (A2 > 0).float()

        # Normalize
        return self.normalize(A2)
    
    def edge_index_to_adj(edge_index, num_nodes):
        A = torch.zeros((num_nodes, num_nodes), device=edge_index.device)
        A[edge_index[0], edge_index[1]] = 1
        return A

    def forward(self, X, edge_index):
        """
        X: (n, in_dim)
        """
        n = X.size(0)

        # Build Adjacency for this Batch
        A = torch.zeros((n,n), device= X.device)
        A[edge_index[0], edge_index[1]] = 1

        # Compute A1 & A2 locally
        A1 = self.normalize(A)
        A2 = self.compute_A2(A)


        # S1: initial embedding
        r0 = F.relu(self.embed(X))  # (n, p)

        representations = [r0]
        r_prev = r0

        # S2: propagation layers
        for k in range(self.K):

            # 1-hop aggregation
            h1 = A1 @ r_prev  # (n, d)

            # 2-hop aggregation
            h2 = A2 @ r_prev  # (n, d)

            # Concatenate (D2)
            r_k = torch.cat([h1, h2], dim=1)  # (n, 2d)

            representations.append(r_k)
            r_prev = r_k  # move to next layer

        # S3: final combination (D3)
        r_final = torch.cat(representations, dim=1)

        out = self.classifier(r_final)

        return out

In [15]:
from torch_geometric.datasets import Planetoid
from torch_geometric.loader import ClusterData, ClusterLoader

dataset = Planetoid(root = "data", name= "cora")
data = dataset[0]

# cluster_data = ClusterData(data, num_parts=1000)
# loader = ClusterLoader(cluster_data, batch_size=5)
data

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])

In [16]:
def train(model, data, optimizer):
  model.train()
  optimizer.zero_grad()
  out = model(data.x, data.edge_index)
  loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
  loss.backward()
  optimizer.step()

  return loss.item()

def evaluate(model, data):
  model.eval()
  out = model(data.x, data.edge_index)
  pred = out.argmax(dim = 1)
  correct = (pred[data.test_mask] == data.y[data.test_mask]).sum().item()
  return correct / data.test_mask.sum().item()

In [28]:
model = H2GCN(
  in_dim= data.num_features,
  hidden_dim=32,
  num_classes=len(data.y.unique()),
  num_layers=2
)

optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

for epoch in range(10):
  loss = train(model, data, optimizer)
  print(f"Epoch {epoch} || Loss : {loss}")
  acc = evaluate(model, data)
  print("accuracy :",acc )

Epoch 0 || Loss : 1.9466694593429565
accuracy : 0.394
Epoch 1 || Loss : 1.8478606939315796
accuracy : 0.545
Epoch 2 || Loss : 1.669877052307129
accuracy : 0.663
Epoch 3 || Loss : 1.4095146656036377
accuracy : 0.722
Epoch 4 || Loss : 1.0970172882080078
accuracy : 0.773
Epoch 5 || Loss : 0.7770376801490784
accuracy : 0.783
Epoch 6 || Loss : 0.5040371417999268
accuracy : 0.78
Epoch 7 || Loss : 0.306084007024765
accuracy : 0.786
Epoch 8 || Loss : 0.17664192616939545
accuracy : 0.796
Epoch 9 || Loss : 0.09790471941232681
accuracy : 0.802
